# Analiza Przestępczości w Chicago z użyciem PySpark

In [3]:
%pip install pyspark
%pip install --upgrade pyspark numpy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [6]:
from datetime import datetime
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder
from pyspark.ml.classification import LogisticRegression

# 1. Inicjalizacja sesji Spark
spark = SparkSession.builder \
    .appName("ChicagoCrimesAnalysis") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/29 21:20:25 WARN Utils: Your hostname, MacBook-Air-Wojciech-2.local, resolves to a loopback address: 127.0.0.1; using 192.168.33.3 instead (on interface en0)
26/05/29 21:20:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/29 21:20:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
# 2. Wczytanie danych z pliku crimes.csv
csv_path = "crimes.csv"

try:
    df_raw = spark.read.csv(csv_path, header=True, inferSchema=True, multiLine=True, escape='"')
    print(f"Wczytano: {csv_path}")
    df_raw.show(3, truncate=False)
except Exception as e:
    print(f"Błąd podczas wczytywania pliku: {e}")

Wczytano: crimes.csv
+--------+-----------+----------------------+----------------------+----+---------------+-----------+-------------------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+----------------------------------+
|ID      |Case Number|Date                  |Block                 |IUCR|Primary Type   |Description|Location Description           |Arrest|Domestic|Beat|District|Ward|Community Area|FBI Code|X Coordinate|Y Coordinate|Year|Updated On            |Latitude    |Longitude    |Location                          |
+--------+-----------+----------------------+----------------------+----+---------------+-----------+-------------------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+----------------------+------------+-------------+----------------------------------+
|14205990|JK264862   |05/21/2026 12:00:00 AM|008XX W

In [8]:
# 3. Czyszczenie danych (Data Cleansing)
def clean_crimes_data(df: DataFrame) -> DataFrame:
    # Parsowanie daty
    df_with_date = df.withColumn(
        "parsed_date", 
        F.to_timestamp(F.col("Date"), "MM/dd/yyyy hh:mm:ss a")
    )
    
    # Filtrowanie błędnych/pustych dat, kluczowych braków danych oraz usuwanie duplikatów po ID biznesowym
    cleaned_df = df_with_date \
        .filter(F.col("parsed_date").isNotNull()) \
        .dropna(subset=["ID", "Primary Type", "Location Description"]) \
        .dropDuplicates(["ID"])
        
    return cleaned_df

df_cleaned = clean_crimes_data(df_raw)

In [9]:
# 4. Definicja i rejestracja UDF (Pora dnia)
def get_time_of_day(dt: datetime) -> str:
    if dt is None:
        return "UNKNOWN"
    hour = dt.hour
    if 5 <= hour < 12:
        return "RANO"
    elif 12 <= hour < 18:
        return "DZIEN"
    elif 18 <= hour < 22:
        return "WIECZOR"
    else:
        return "NOC"

# Rejestracja funkcji UDF z jawnym typem zwracanym
time_of_day_udf = F.udf(get_time_of_day, StringType())

# Dodanie kolumny do zbioru danych
df_with_time = df_cleaned.withColumn("Pora_Dnia", time_of_day_udf(F.col("parsed_date")))
df_with_time.select("Date", "parsed_date", "Pora_Dnia").show(5, truncate=False)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pyspark/sql/udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


+----------------------+-------------------+---------+
|Date                  |parsed_date        |Pora_Dnia|
+----------------------+-------------------+---------+
|01/03/2026 09:27:00 AM|2026-01-03 09:27:00|RANO     |
|01/04/2026 10:21:00 PM|2026-01-04 22:21:00|NOC      |
|01/06/2026 06:24:00 PM|2026-01-06 18:24:00|WIECZOR  |
|01/07/2026 05:17:00 PM|2026-01-07 17:17:00|DZIEN    |
|01/08/2026 08:46:00 AM|2026-01-08 08:46:00|RANO     |
+----------------------+-------------------+---------+
only showing top 5 rows


In [10]:
# 5. Optymalizacja: Broadcast Join & Cache
fbi_data = [("14", "Property Damage"), ("06", "Larceny"), ("26", "Trespass"), ("08B", "Assault")]
lookup_df = spark.createDataFrame(fbi_data, ["FBI Code", "FBI_Category"])

df_enriched = df_with_time.join(F.broadcast(lookup_df), "FBI Code", "left")

# Persystencja danych w pamięci (Cache)
df_enriched.cache()
print("Zbiór danych został wzbogacony i zapisany w pamięci podręcznej (cached).")

Zbiór danych został wzbogacony i zapisany w pamięci podręcznej (cached).


26/05/29 21:21:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [11]:
# 6. Zapis wynikowych danych do formatu Parquet z podziałem na partycje po roku
output_parquet_path = "output_chicago_crimes"

df_enriched.write \
    .mode("overwrite") \
    .partitionBy("Year") \
    .parquet(output_parquet_path)

print(f"Dane zostały zapisane w formacie Parquet w lokalizacji: {output_parquet_path}")

26/05/29 21:21:43 WARN MemoryManager: Total allocation exceeds 95,00% (1 020 054 720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


Dane zostały zapisane w formacie Parquet w lokalizacji: output_chicago_crimes


In [12]:
# 7. Analiza statystyczna przestępstw wraz z wizualizacją planu zapytań (.explain)

print("=== ANALIZA 1: Przestępstwa według typu (Top 5) ===")
stats_type = df_enriched.groupBy("Primary Type").count().orderBy(F.desc("count"))
stats_type.explain(extended=True)
stats_type.show(5)

print("\n=== ANALIZA 2: Przestępstwa według lokalizacji (Top 5) ===")
stats_loc = df_enriched.groupBy("Location Description").count().orderBy(F.desc("count"))
stats_loc.explain(mode="formatted")
stats_loc.show(5)

print("\n=== ANALIZA 3: Przestępstwa według pory dnia ===")
stats_time = df_enriched.groupBy("Pora_Dnia").count().orderBy(F.desc("count"))
stats_time.explain(mode="simple")
stats_time.show()

=== ANALIZA 1: Przestępstwa według typu (Top 5) ===
== Parsed Logical Plan ==
'Sort ['count DESC NULLS LAST], true
+- Aggregate [Primary Type#5], [Primary Type#5, count(1) AS count#1018L]
   +- Project [FBI Code#14, ID#0, Case Number#1, Date#2, Block#3, IUCR#4, Primary Type#5, Description#6, Location Description#7, Arrest#8, Domestic#9, Beat#10, District#11, Ward#12, Community Area#13, X Coordinate#15, Y Coordinate#16, Year#17, Updated On#18, Latitude#19, Longitude#20, Location#21, parsed_date#112, Pora_Dnia#114, FBI_Category#191]
      +- Join LeftOuter, (FBI Code#14 = FBI Code#190)
         :- Project [ID#0, Case Number#1, Date#2, Block#3, IUCR#4, Primary Type#5, Description#6, Location Description#7, Arrest#8, Domestic#9, Beat#10, District#11, Ward#12, Community Area#13, FBI Code#14, X Coordinate#15, Y Coordinate#16, Year#17, Updated On#18, Latitude#19, Longitude#20, Location#21, parsed_date#112, get_time_of_day(parsed_date#112)#113 AS Pora_Dnia#114]
         :  +- Deduplicate [ID#0

+-------------------+-----+
|       Primary Type|count|
+-------------------+-----+
|              THEFT|18358|
|            BATTERY|15427|
|    CRIMINAL DAMAGE| 9072|
|            ASSAULT| 7610|
|MOTOR VEHICLE THEFT| 6629|
+-------------------+-----+
only showing top 5 rows

=== ANALIZA 2: Przestępstwa według lokalizacji (Top 5) ===
== Physical Plan ==
AdaptiveSparkPlan (39)
+- Sort (38)
   +- Exchange (37)
      +- HashAggregate (36)
         +- Exchange (35)
            +- HashAggregate (34)
               +- InMemoryTableScan (1)
                     +- InMemoryRelation (2)
                           +- AdaptiveSparkPlan (33)
                              +- == Final Plan ==
                                 ResultQueryStage (20)
                                 +- * Project (19)
                                    +- * BroadcastHashJoin LeftOuter BuildRight (18)
                                       :- * Project (13)
                                       :  +- BatchEvalPython (12

+--------------------+-----+
|Location Description|count|
+--------------------+-----+
|              STREET|23139|
|           APARTMENT|16958|
|           RESIDENCE| 9662|
|            SIDEWALK| 3642|
|PARKING LOT / GAR...| 3069|
+--------------------+-----+
only showing top 5 rows

=== ANALIZA 3: Przestępstwa według pory dnia ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [count#3090L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(count#3090L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=486]
      +- HashAggregate(keys=[Pora_Dnia#114], functions=[count(1)])
         +- Exchange hashpartitioning(Pora_Dnia#114, 200), ENSURE_REQUIREMENTS, [plan_id=483]
            +- HashAggregate(keys=[Pora_Dnia#114], functions=[partial_count(1)])
               +- InMemoryTableScan [Pora_Dnia#114]
                     +- InMemoryRelation [FBI Code#14, ID#0, Case Number#1, Date#2, Block#3, IUCR#4, Primary Type#5, Description#6, Location Description#7, Arrest#8,

In [13]:
# 8. Budowa modelu klasyfikacji typów przestępstw za pomocą MLlib (Pipeline)

# Przygotowanie ramki danych z cechami i wyodrębnieniem godziny jako zmiennej numerycznej
ml_df = df_enriched.withColumn("Hour", F.hour(F.col("parsed_date"))).select("Primary Type", "Hour", "Location Description", "Pora_Dnia")

# Transformery cech kategorycznych
indexer_loc = StringIndexer(inputCol="Location Description", outputCol="Loc_Index", handleInvalid="skip")
indexer_pora = StringIndexer(inputCol="Pora_Dnia", outputCol="Pora_Index", handleInvalid="skip")

# OneHotEncoder dla zmiennych kategorycznych
encoder = OneHotEncoder(
    inputCols=["Loc_Index", "Pora_Index"],
    outputCols=["Loc_Vec", "Pora_Vec"]
)

# Indeksowanie etykiety docelowej (Primary Type)
label_indexer = StringIndexer(inputCol="Primary Type", outputCol="label", handleInvalid="skip")

# Agregacja wszystkich cech (numerycznych i zakodowanych wektorów) w jeden wektor wejściowy
assembler = VectorAssembler(
    inputCols=["Hour", "Loc_Vec", "Pora_Vec"],
    outputCol="features"
)

# Klasyfikator - Wieloklasowa Regresja Logistyczna
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)

# Budowa pipeline'u
pipeline = Pipeline(stages=[indexer_loc, indexer_pora, encoder, label_indexer, assembler, lr])

# Podział danych na zbiór treningowy (80%) i testowy (20%)
train_data, test_data = ml_df.randomSplit([0.8, 0.2], seed=42)

print("Rozpoczęcie trenowania modelu wewnątrz Pipeline...")
model = pipeline.fit(train_data)
print("Model został pomyślnie wytrenowany.")

# Predykcja na danych testowych
predictions = model.transform(test_data)
print("\nPrzykładowe wyniki klasyfikacji modelu:")
predictions.select("Primary Type", "label", "prediction", "probability").show(5, truncate=False)

Rozpoczęcie trenowania modelu wewnątrz Pipeline...


26/05/29 21:22:05 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Model został pomyślnie wytrenowany.

Przykładowe wyniki klasyfikacji modelu:
+------------+-----+----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Primary Type|label|prediction|probability                                                                                                                                                                                                                        

In [14]:
# 9. Zamknięcie sesji Spark i czyszczenie cache
df_enriched.unpersist()
spark.stop()
print("Sesja Spark została zamknięta")

Sesja Spark została zamknięta
